# 🤖 Fine-tune Qwen2.5:3B on Frontiir Data
**Method:** QLoRA with Unsloth  
**GPU:** T4 (free Colab)  
**Output:** GGUF → Ollama

| Step | Description |
|------|-------------|
| 1 | Check GPU |
| 2 | Install Unsloth |
| 3 | Load base model |
| **4** | **📊 BEFORE training status** |
| 5 | Apply QLoRA |
| 6 | Upload train.jsonl |
| 7 | Format data |
| 8 | Train |
| **9** | **📊 AFTER training status + comparison** |
| 10 | Export GGUF |
| 11 | Download to PC |

## ✅ Step 1 — Check GPU

In [ ]:
!nvidia-smi

## ✅ Step 2 — Install Unsloth & Dependencies

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q

import pathlib

# ── Patch 1: unsloth/models/rl.py — sanitize_logprob KeyError ──
rl_path = pathlib.Path("/usr/local/lib/python3.12/dist-packages/unsloth/models/rl.py")
if rl_path.exists():
    c = rl_path.read_text()
    if 'RL_REPLACEMENTS["sanitize_logprob"]' in c:
        c = c.replace(
            'sanitize_logprob = RL_REPLACEMENTS["sanitize_logprob"]',
            'sanitize_logprob = RL_REPLACEMENTS.get("sanitize_logprob", None)',
        )
        rl_path.write_text(c)
        print("✅ Patch 1 applied: rl.py sanitize_logprob")
    else:
        print("✅ Patch 1: rl.py already fine")

# ── Patch 2: trl/trainer/utils.py — entropy_from_logits ──
# Append a safe replacement at the END of the file so Python's last-definition
# rule makes this the active version — even if unsloth re-reads the file later.
utils_path = pathlib.Path("/usr/local/lib/python3.12/dist-packages/trl/trainer/utils.py")
if utils_path.exists():
    c = utils_path.read_text()
    if '_ENTROPY_PATCHED_BY_FRONTIIR' not in c:
        safe_fn = '''

# ── Appended by train_qwen.ipynb — fixes unsloth monkey-patch conflict ──
_ENTROPY_PATCHED_BY_FRONTIIR = True

def entropy_from_logits(_logits, _chunk_size=1):
    """Safe re-implementation with renamed args to avoid unsloth exec() collision."""
    import torch as _torch
    _shape = _logits.shape[:-1]
    _ncls  = _logits.shape[-1]
    _flat  = _logits.reshape(-1, _ncls)
    _out   = _torch.zeros(_flat.shape[0], device=_flat.device, dtype=_flat.dtype)
    for _i in range(0, _flat.shape[0], _chunk_size):
        _c = _flat[_i : _i + _chunk_size]
        _out[_i : _i + _chunk_size] = -(
            _torch.softmax(_c, dim=-1) * _torch.log_softmax(_c, dim=-1)
        ).sum(dim=-1)
    return _out.reshape(_shape)
'''
        utils_path.write_text(c + safe_fn)
        print("✅ Patch 2 applied: trl/trainer/utils.py entropy_from_logits")
    else:
        print("✅ Patch 2: trl utils.py already patched")
else:
    print("⚠️  trl/trainer/utils.py not found — skip patch 2")

print()
print("✅ All patches done — safe to import unsloth")

## ✅ Step 3 — Load Qwen2.5-3B Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print('✅ Qwen2.5-3B loaded')

## 📊 Step 4 — BEFORE Training Status
> Tests the **base model** (no fine-tuning yet) on Frontiir questions.  
> Results are saved and compared after training.

In [ ]:
import time

SYSTEM_PROMPT = """You are Frontiir AI Assistant — a helpful customer support assistant for Frontiir, Myanmar's leading fiber internet provider. Answer questions about internet packages, billing, technical support, and company information. Respond in the same language the user uses (Burmese or English)."""

# ── Evaluation questions (mix of Burmese and English) ──
EVAL_QUESTIONS = [
    ("EN", "What internet packages does Frontiir offer?"),
    ("EN", "How do I contact customer service?"),
    ("EN", "Why is my internet slow?"),
    ("EN", "When was Frontiir founded?"),
    ("MM", "Frontiir internet package တွေ ဘာတွေ ရှိလဲ"),
    ("MM", "Customer service ဆက်သွယ်နည်း"),
    ("MM", "Internet မကောင်းရင် ဘာလုပ်ရမလဲ"),
    ("MM", "Payment ဘယ်လို ပေးရမလဲ"),
]

def ask_model(mdl, tkn, question, max_new_tokens=200):
    """Run inference and return (answer, time_sec)"""
    # ✅ FIX 1: for_inference() moved OUT of this function
    #           (call it once before the loop, not 8 times)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    # ✅ FIX 2 & 3: apply_chat_template with tokenize=False first,
    #               then tokenize separately → always returns a plain tensor
    text = tkn.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tkn(text, return_tensors="pt").to("cuda")
    input_ids = inputs.input_ids  # plain tensor — safe for .shape[1]

    t0 = time.time()
    with torch.no_grad():
        outputs = mdl.generate(
            input_ids=input_ids,                    # ✅ plain tensor, not BatchEncoding
            attention_mask=inputs.attention_mask,   # ✅ pass attention mask too
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tkn.eos_token_id,
        )
    elapsed = time.time() - t0
    answer = tkn.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)  # ✅ fixed
    return answer.strip(), round(elapsed, 2)

def score_answer(answer, question):
    """Simple scoring: 0-5 based on answer quality signals."""
    answer_lower = answer.lower()
    score = 0
    if len(answer) > 30:  score += 1
    if len(answer) > 80:  score += 1
    if 'frontiir' in answer_lower: score += 1
    keywords = ['package', 'internet', 'service', 'customer', 'support',
                'contact', 'kyat', 'mmk', 'payment', 'fiber', 'mbps',
                'ကျပ်', 'ဆက်သွယ်']  # ✅ removed duplicates
    if any(k in answer_lower for k in keywords): score += 1
    refusals = ["i don't know", "i cannot", "i'm not sure", "no information"]
    if not any(r in answer_lower for r in refusals): score += 1
    return min(score, 5)

print('='*65)
print('  📊 BEFORE TRAINING — Base Model Evaluation')
print('='*65)
print(f'  Model : Qwen2.5-3B-Instruct (no fine-tuning)')
print(f'  Questions: {len(EVAL_QUESTIONS)}')
print('='*65)

before_results = []
total_score_before = 0

# ✅ FIX 1: Call for_inference() ONCE here, before the loop
FastLanguageModel.for_inference(model)

for i, (lang, question) in enumerate(EVAL_QUESTIONS, 1):
    print(f'\n[{i}/{len(EVAL_QUESTIONS)}] [{lang}] {question}')
    answer, elapsed = ask_model(model, tokenizer, question)
    sc = score_answer(answer, question)
    total_score_before += sc
    before_results.append({
        'lang': lang, 'question': question,
        'answer': answer, 'time': elapsed, 'score': sc
    })
    stars = '⭐' * sc + '☆' * (5 - sc)
    print(f'  Answer : {answer[:120]}...' if len(answer) > 120 else f'  Answer : {answer}')
    print(f'  Score  : {stars} ({sc}/5)  |  Time: {elapsed}s')

avg_before = total_score_before / len(EVAL_QUESTIONS)
print()
print('='*65)
print(f'  BEFORE TRAINING SUMMARY')
print(f'  Total Score : {total_score_before}/{len(EVAL_QUESTIONS)*5}')
print(f'  Average     : {avg_before:.1f}/5.0')
grade = 'Excellent' if avg_before>=4.5 else 'Good' if avg_before>=3.5 else 'Fair' if avg_before>=2.5 else 'Poor'
print(f'  Grade       : {grade}')
print('='*65)
print('  ✅ Before-training results saved. Proceed to next steps.')

## ✅ Step 5 — Apply QLoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print('✅ QLoRA adapters applied')
print(f'   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## ✅ Step 6 — Upload train.jsonl
> Upload your `train.jsonl` file from your PC

In [ ]:
import os, shutil

# ── Detect environment ──
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    # ── Colab: upload file via browser ──
    from google.colab import files
    print('📁 Upload your train.jsonl file...')
    uploaded = files.upload()
    print('✅ File uploaded:', list(uploaded.keys()))
else:
    # ── Local: file should already be in project folder ──
    LOCAL_JSONL = os.path.join(os.path.dirname(os.path.abspath('train_qwen.ipynb')), 'train.jsonl')
    if os.path.exists('train.jsonl'):
        print('✅ Found train.jsonl in current directory')
    elif os.path.exists(LOCAL_JSONL):
        shutil.copy(LOCAL_JSONL, 'train.jsonl')
        print(f'✅ Copied train.jsonl from {LOCAL_JSONL}')
    else:
        print('❌ train.jsonl not found!')
        print('   Put train.jsonl in the same folder as this notebook and re-run.')

## ✅ Step 7 — Format Training Data

In [ ]:
import json
from datasets import Dataset

data = []
with open('train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f'✅ Loaded {len(data)} training examples')

def format_chat(example):
    instruction = example['instruction']
    inp = example.get('input', '')
    output = example['output']
    user_msg = f"{instruction}\n{inp}" if inp and inp.strip() else instruction
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": output},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

raw_dataset = Dataset.from_list(data)
dataset = raw_dataset.map(format_chat, remove_columns=raw_dataset.column_names)

print(f'✅ Dataset formatted: {len(dataset)} examples')

## ✅ Step 8 — Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "./frontiir-qwen-outputs",
        report_to = "none",
    ),
)

print('🚀 Starting training...')
print(f'   Dataset : {len(dataset)} examples')
print(f'   Epochs  : 3')
print(f'   ETA     : ~10-20 min on T4 GPU')
print()

trainer_stats = trainer.train()

print()
print('✅ Training complete!')
print(f'   Steps : {trainer_stats.global_step}')
print(f'   Loss  : {trainer_stats.training_loss:.4f}')

## 📊 Step 9 — AFTER Training Status + Comparison
> Tests the **fine-tuned model** on the same questions and shows a side-by-side comparison.

In [ ]:
# ── Plot training loss curve ──
try:
    import matplotlib.pyplot as plt

    log_history = trainer.state.log_history
    steps  = [x['step'] for x in log_history if 'loss' in x]
    losses = [x['loss'] for x in log_history if 'loss' in x]

    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, 'b-o', markersize=4, linewidth=2)
    plt.title('Training Loss Curve — Frontiir QLoRA Fine-tuning', fontsize=13)
    plt.xlabel('Training Steps')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_loss.png', dpi=120)
    plt.show()
    print(f'📉 Loss: {losses[0]:.4f} → {losses[-1]:.4f}  (decreased by {losses[0]-losses[-1]:.4f})')
except Exception as e:
    print(f'Plot skipped: {e}')

In [ ]:
# ── Run the same questions on the fine-tuned model ──
print('='*65)
print('  📊 AFTER TRAINING — Fine-tuned Model Evaluation')
print('='*65)
print(f'  Model    : Qwen2.5-3B + Frontiir QLoRA')
print(f'  Training loss: {trainer_stats.training_loss:.4f}')
print(f'  Questions: {len(EVAL_QUESTIONS)}')
print('='*65)

after_results = []
total_score_after = 0

# ✅ Must call for_inference() again after training
#    (trainer.train() switches model back to training mode)
FastLanguageModel.for_inference(model)

for i, (lang, question) in enumerate(EVAL_QUESTIONS, 1):
    print(f'\n[{i}/{len(EVAL_QUESTIONS)}] [{lang}] {question}')
    answer, elapsed = ask_model(model, tokenizer, question)
    sc = score_answer(answer, question)
    total_score_after += sc
    after_results.append({
        'lang': lang, 'question': question,
        'answer': answer, 'time': elapsed, 'score': sc
    })
    stars = '⭐' * sc + '☆' * (5 - sc)
    print(f'  Answer : {answer[:120]}...' if len(answer) > 120 else f'  Answer : {answer}')
    print(f'  Score  : {stars} ({sc}/5)  |  Time: {elapsed}s')

avg_after = total_score_after / len(EVAL_QUESTIONS)

In [ ]:
# ── Side-by-side comparison table ──
print()
print('='*65)
print('  📊 BEFORE vs AFTER — Full Comparison')
print('='*65)

max_q = 45
print(f"  {'Question':<{max_q}}  Before  After  Change")
print(f"  {'-'*max_q}  ------  -----  ------")

for b, a in zip(before_results, after_results):
    q_short = b['question'][:max_q-2] + '..' if len(b['question']) > max_q else b['question']
    diff = a['score'] - b['score']
    arrow = f'↑ +{diff}' if diff > 0 else (f'↓ {diff}' if diff < 0 else '→  0')
    b_stars = '⭐' * b['score']
    a_stars = '⭐' * a['score']
    print(f"  [{b['lang']}] {q_short:<{max_q}}  {b['score']}/5   {a['score']}/5   {arrow}")

print(f"  {'-'*max_q}  ------  -----  ------")
diff_total = total_score_after - total_score_before
diff_avg = avg_after - avg_before
arrow_total = f'↑ +{diff_total}' if diff_total > 0 else (f'↓ {diff_total}' if diff_total < 0 else '→  0')

print(f"  {'TOTAL':<{max_q}}  {total_score_before}/{len(EVAL_QUESTIONS)*5}  {total_score_after}/{len(EVAL_QUESTIONS)*5}  {arrow_total}")
print(f"  {'AVERAGE':<{max_q}}  {avg_before:.1f}/5  {avg_after:.1f}/5  {'+' if diff_avg >= 0 else ''}{diff_avg:.1f}")

print()
print('='*65)
print('  FINAL RESULT')
print('='*65)

grade_before = 'Excellent' if avg_before>=4.5 else 'Good' if avg_before>=3.5 else 'Fair' if avg_before>=2.5 else 'Poor'
grade_after  = 'Excellent' if avg_after>=4.5  else 'Good' if avg_after>=3.5  else 'Fair' if avg_after>=2.5  else 'Poor'

print(f'  Before Training  : {avg_before:.1f}/5.0  →  {grade_before}')
print(f'  After  Training  : {avg_after:.1f}/5.0  →  {grade_after}')
print()

if diff_avg > 0.5:
    print('  ✅ Training significantly improved the model!')
elif diff_avg > 0:
    print('  ✅ Training improved the model.')
elif diff_avg == 0:
    print('  ➡️  Model performance unchanged (may need more epochs or data).')
else:
    print('  ⚠️  Model regressed — try reducing learning rate or more data.')

print(f'  Training Loss    : {trainer_stats.training_loss:.4f}')
print('  (Lower loss = better learning from your data)')
print('='*65)

In [ ]:
# ── Bar chart: Before vs After scores per question ──
import matplotlib.pyplot as plt
import numpy as np

questions_short = [
    f"[{b['lang']}] {b['question'][:30]}..." if len(b['question']) > 30
    else f"[{b['lang']}] {b['question']}"
    for b in before_results
]
scores_before = [r['score'] for r in before_results]
scores_after  = [r['score'] for r in after_results]

x = np.arange(len(questions_short))
width = 0.38

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, scores_before, width, label='Before Training', color='#ff7f7f', alpha=0.85)
bars2 = ax.bar(x + width/2, scores_after,  width, label='After Training',  color='#7fbf7f', alpha=0.85)

ax.set_ylabel('Score (out of 5)')
ax.set_title('Frontiir AI — Before vs After Fine-tuning Quality', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(questions_short, rotation=35, ha='right', fontsize=8)
ax.set_ylim(0, 6)
ax.legend(fontsize=11)
ax.axhline(y=avg_before, color='red',   linestyle='--', alpha=0.5, label=f'Avg Before: {avg_before:.1f}')
ax.axhline(y=avg_after,  color='green', linestyle='--', alpha=0.5, label=f'Avg After:  {avg_after:.1f}')
ax.legend(fontsize=10)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{bar.get_height()}', ha='center', va='bottom', fontsize=9, color='darkred')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{bar.get_height()}', ha='center', va='bottom', fontsize=9, color='darkgreen')

plt.tight_layout()
plt.savefig('before_after_comparison.png', dpi=130)
plt.show()
print('✅ Chart saved as before_after_comparison.png')

In [ ]:
# ── Save full report as text file ──
import datetime

report_lines = [
    '='*65,
    '  FRONTIIR AI — TRAINING REPORT',
    f'  Date: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'  Model: Qwen2.5-3B-Instruct + QLoRA',
    f'  Training Loss: {trainer_stats.training_loss:.4f}',
    '='*65,
    '',
    f'BEFORE Training Score: {total_score_before}/{len(EVAL_QUESTIONS)*5}  ({avg_before:.1f}/5.0)  [{grade_before}]',
    f'AFTER  Training Score: {total_score_after}/{len(EVAL_QUESTIONS)*5}  ({avg_after:.1f}/5.0)  [{grade_after}]',
    f'Improvement          : {diff_avg:+.1f} points',
    '',
    '-'*65,
    'DETAILED RESULTS',
    '-'*65,
]

for i, (b, a) in enumerate(zip(before_results, after_results), 1):
    report_lines += [
        f'',
        f'Q{i} [{b["lang"]}]: {b["question"]}',
        f'  BEFORE ({b["score"]}/5): {b["answer"][:200]}',
        f'  AFTER  ({a["score"]}/5): {a["answer"][:200]}',
    ]

report_text = '\n'.join(report_lines)
with open('training_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)

print('✅ Full report saved: training_report.txt')
print()
print(report_text[:800], '...')

## ✅ Step 10 — Export to GGUF (for Ollama)
> Q4_K_M quantization — best balance of quality and size (~2GB)

In [ ]:
print('💾 Saving LoRA adapter...')
model.save_pretrained("frontiir-qwen-lora")
tokenizer.save_pretrained("frontiir-qwen-lora")
print('✅ LoRA adapter saved')

print()
print('📦 Exporting to GGUF Q4_K_M (~5 min)...')
model.save_pretrained_gguf(
    "frontiir-qwen-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)
print('✅ GGUF export complete!')

import os
gguf_files = [f for f in os.listdir('frontiir-qwen-gguf') if f.endswith('.gguf')]
gguf_path = f'frontiir-qwen-gguf/{gguf_files[0]}'
size_mb = os.path.getsize(gguf_path) / (1024*1024)
print(f'   File: {gguf_path}  ({size_mb:.0f} MB)')

## ✅ Step 11 — Download Files to your PC

In [ ]:
import os

IN_COLAB = 'google.colab' in str(get_ipython())

gguf_dir  = 'frontiir-qwen-gguf'
gguf_file = [f for f in os.listdir(gguf_dir) if f.endswith('.gguf')][0]
gguf_path = os.path.join(gguf_dir, gguf_file)
size_mb   = os.path.getsize(gguf_path) / (1024 * 1024)

if IN_COLAB:
    # ── Step A: Save to Google Drive first (safe backup) ──
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        save_dir = '/content/drive/MyDrive/frontiir-ai'
        os.makedirs(save_dir, exist_ok=True)

        import shutil
        shutil.copy(gguf_path,                    f'{save_dir}/{gguf_file}')
        shutil.copy('training_report.txt',         f'{save_dir}/training_report.txt')
        if os.path.exists('before_after_comparison.png'):
            shutil.copy('before_after_comparison.png', f'{save_dir}/before_after_comparison.png')
        if os.path.exists('training_loss.png'):
            shutil.copy('training_loss.png',           f'{save_dir}/training_loss.png')

        print('✅ Files saved to Google Drive: MyDrive/frontiir-ai/')
        print(f'   📦 {gguf_file}  ({size_mb:.0f} MB)')
        print(f'   📄 training_report.txt')
        print()
    except Exception as e:
        print(f'⚠️  Drive save failed: {e}')

    # ── Step B: Also download directly to PC ──
    from google.colab import files
    print(f'📥 Downloading to PC (browser download)...')
    print(f'   {gguf_file}  ({size_mb:.0f} MB) — may take a few minutes')
    files.download(gguf_path)
    files.download('training_report.txt')
    if os.path.exists('before_after_comparison.png'):
        files.download('before_after_comparison.png')
    if os.path.exists('training_loss.png'):
        files.download('training_loss.png')
    print('✅ Browser downloads triggered!')

else:
    # ── Local: files already on disk ──
    print('✅ Files saved locally:')
    print(f'   📦 {os.path.abspath(gguf_path)}  ({size_mb:.0f} MB)')
    print(f'   📄 {os.path.abspath("training_report.txt")}')

print()
print('Next steps on your PC:')
print(f'  bash train.sh ~/Downloads/{gguf_file}')